# Classify Product Type

### Environment
Ubuntu 22.04 LTS which includes **Python 3.9.12** and utilities *curl*, *git*, *vim*, *unzip*, *wget*, and *zip*. There is no *GPU* support.

The IPython Kernel allows you to execute Python code in the Notebook cell and Python console.

### Installing packages
- Run `!mamba list "package_name"` command to check the package installation status.
- Run the `!mamba install "package_name"` to install a package

In [ ]:
# Install TensorFlow if it is not available in the environment
!mamba install -c conda-forge tensorflow -y

In [ ]:
# Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

pd.set_option("display.max_columns", 101)
pd.set_option("display.max_colwidth", None)

## Data Description

| Column    | Description                                         |
|:----------|:----------------------------------------------------|
| `desc`  | description of product                                |
| `label`   | Product Type (0-Movie DVD, 1-Electronics, 2-Kitchen Appliances) |

In [ ]:
# The training dataset is already loaded below.
data = pd.read_csv("train.csv")
data.head()

In [ ]:
# The testing dataset is already loaded below.
test = pd.read_csv("test.csv")
test.head()

## Text Preprocessing

We will tokenize the descriptions and pad them to ensure uniform input length for the neural network.
- **Vocabulary size:** 10,000 most common words
- **Max sequence length:** 150 tokens

In [ ]:
# Configure tokenizer
MAX_WORDS = 10000
MAX_LEN = 150

tokenizer = Tokenizer(num_words=MAX_WORDS)
tokenizer.fit_on_texts(data['desc'])

# Convert text to sequences
train_sequences = tokenizer.texts_to_sequences(data['desc'])
test_sequences = tokenizer.texts_to_sequences(test['desc'])

# Pad sequences
train_padded = pad_sequences(train_sequences, maxlen=MAX_LEN, padding='post')
test_padded = pad_sequences(test_sequences, maxlen=MAX_LEN, padding='post')

## Deep Learning Model

Build a neural network that can classify the product types.
- **The model's performance will be evaluated on the basis of accuracy score.**
- Model architecture: Embedding layer -> Global Average Pooling -> Dense layers.
- Early stopping is used during training to prevent overfitting.

In [ ]:
# Build the model
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(MAX_WORDS, 16, input_length=MAX_LEN),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.5),  # Prevents overfitting
    tf.keras.layers.Dense(3, activation='softmax')  # 3 output classes
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

In [ ]:
# Train the model with EarlyStopping
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = model.fit(train_padded, data['label'],
                    epochs=30,
                    batch_size=32,
                    validation_split=0.2,
                    callbacks=[early_stop],
                    verbose=1)

In [ ]:
# Assess model performance on train.csv
loss, accuracy = model.evaluate(train_padded, data['label'], verbose=0)
print(f"Final Training Accuracy: {accuracy:.4f}")

In [ ]:
# Plot training history to check for overfitting
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Model Accuracy')
plt.show()

> #### Task:
- **Submit the predictions on the test dataset using your optimized model** <br/>
    For each record in the test set (test.csv), predict the value of the `label` variable.  You should submit a CSV file with a header row and one row per test entry.

The file (`submissions.csv`) should have exactly 2 columns:

| Column    | Description                                         |
|:----------|:----------------------------------------------------|
| `desc`  | description of product                                |
| `label`   | Product Type (0-Movie DVD, 1-Electronics, 2-Kitchen Appliances) |

In [ ]:
# Prediction and Submission
predictions = model.predict(test_padded)
predicted_classes = np.argmax(predictions, axis=1)

# Create the submission dataframe with correct column names and format
submission_df = pd.DataFrame({
    'desc': test['desc'],
    'label': predicted_classes
})

# Save as submissions.csv without the index
submission_df.to_csv("submissions.csv", index=False)

print("File 'submissions.csv' successfully generated and saved.")
submission_df.head()